# M0: LARRY data check

未実行。公式 Quickstart の subset を再現する。実データの数・shape は実行結果から記録する。

Source: https://www.scdiffeq.com/_tutorials/quickstart.html

専用 Python 3.11 環境と保存先を用意し、`BIO_SDE_DATA_DIR` を指定する。キャッシュの二重保存・前処理を含め 20 GB 以上の空きを確保する。

In [ ]:
from pathlib import Path
import os
import sys
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "1007_instruction.md").exists()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
DATA_DIR = Path(os.environ.get("BIO_SDE_DATA_DIR", "data"))
SEED = 0


In [ ]:
import json
from src.preflight import inspect_environment
status = inspect_environment(DATA_DIR)
Path("outputs/logs/preflight.json").write_text(json.dumps(status, indent=2))
print(json.dumps(status, indent=2))
assert status["ready"], "環境・容量を整えてから再実行してください"


In [ ]:
import lightning as pl
import scdiffeq as sdq
from src.data import quickstart_subset, save_inventory
pl.seed_everything(SEED, workers=True)
adata_ref = sdq.datasets.larry(data_dir=str(DATA_DIR), variant=None)
adata = quickstart_subset(adata_ref)
summary = save_inventory(adata_ref, adata, Path("outputs"))
summary


In [ ]:
import pandas as pd
from IPython.display import display
for label in ("full", "quickstart_subset"):
    display(pd.read_csv(f"outputs/tables/{label}_time_by_class.csv"))


## 解釈上の制約

付属 PCA/scaling は upstream 全データを使っている可能性がある。公式再現と独立 test 評価は区別する。次段階の分割では clone の重複・前処理 fit 範囲を監査し、test を library/threshold tuning に使わない。UMAP の外観のみで SDE の妥当性を判断しない。

In [ ]:
import subprocess
from src.data import sha256_file
cache = DATA_DIR / "scdiffeq_data" / "larry"
manifest = [{"path": str(p), "bytes": p.stat().st_size, "sha256": sha256_file(p)}
            for p in sorted(cache.glob("*")) if p.is_file()]
Path("outputs/logs/data_provenance.json").write_text(json.dumps(
    {"seed": SEED, "variant": None, "preflight": status, "files": manifest}, indent=2))
Path("requirements-lock.txt").write_text(subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True))
